In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
# Imports and env setup

from sklearn.model_selection import train_test_split
import pandas as pd
print(f"Input: {MACHINE_A_DATASET_FILE}")

In [ ]:
# load dataset

print("Loading full dataset...")
df = pd.read_csv(MACHINE_A_DATASET_FILE)

print(f"Total Samples: {len(df)}")

In [ ]:
# Split into (Train + Val) and (Test)
# We reserve 15% for the final Hold-out Test Set
train_val_df, test_df = train_test_split(
    df, 
    test_size=0.15, 
    random_state=42, 
    stratify=df['label']
)

# Split (Train + Val) into (Train) and (Val)
# We need 15% of the TOTAL to be validation. 
# Remaining is 85%. 15 / 85 ~= 0.176
val_size_adjusted = 0.15 / 0.85

train_df, val_df = train_test_split(
    train_val_df, 
    test_size=val_size_adjusted, 
    random_state=42, 
    stratify=train_val_df['label']
)

print(f"--- Split Sizes ---")
print(f"Train Set: {len(train_df)} ({(len(train_df)/len(df))*100:.1f}%)")
print(f"Val Set:   {len(val_df)}   ({(len(val_df)/len(df))*100:.1f}%)")
print(f"Test Set:  {len(test_df)}  ({(len(test_df)/len(df))*100:.1f}%)")

In [ ]:
# verify class balance

def check_balance(name, dataset):
    count = dataset['label'].value_counts(normalize=True)
    print(f"[{name}] Clean: {count.get(0, 0):.2%}, Jammed: {count.get(1, 0):.2%}")

print("--- Class Balance Verification ---")
check_balance("Original", df)
check_balance("Train   ", train_df)
check_balance("Val     ", val_df)
check_balance("Test    ", test_df)

In [ ]:
# save files

print("Saving to CSV...")

train_df.to_csv(MACHINE_A_TRAIN_SET_FILE, index=False)
val_df.to_csv(MACHINE_A_VAL_SET_FILE, index=False)
test_df.to_csv(MACHINE_A_TEST_SET_FILE, index=False)

print("Splitting Complete.")
print(f"Train: {MACHINE_A_TRAIN_SET_FILE}")
print(f"Val:   {MACHINE_A_VAL_SET_FILE}")
print(f"Test:  {MACHINE_A_TEST_SET_FILE}")